In [2]:
corpus = [
    "low roll fox",
    "lower"
]

def normalize_text(text):
    normal = text.strip().lower()
    return normal

def split_sentence(text):
    word_list = text.split()
    return word_list

def toCharacters(word):
    return list(word) + ['</w>']

def normalize(corpus):
    processed = []

    for sentence in corpus:
        sentence = normalize_text(sentence)
        words = split_sentence(sentence)
        sentence_list = []
        for word in words:
            sentence_list.append(toCharacters(word))

        processed.append(sentence_list)

    return processed


In [3]:
def corpus_to_word_frequency(corpus):

    word_frequency = dict()

    for sentence in corpus:
        for word in sentence:
            if tuple(word) in word_frequency:
                word_frequency[tuple(word)] += 1
            else:
                word_frequency[tuple(word)] = 1
    return word_frequency

In [4]:
def generate_initial_vocab(word_freq):
    vocab = []
    for word in word_freq:
        for character in word:
            if character not in vocab:
                vocab.append(character)
    return vocab

In [5]:
def count_pairs(word_freq):
    pairs = dict()
    for word in word_freq:
        for i in range (len(word) -1):
            pair = (word[i], word[i+1])
            
            if pair in pairs:
                pairs[pair] += word_freq[word]
            else:
                pairs[pair] = word_freq[word]
    return pairs
                    
def get_frequent_pair(pairs):
    max_pair = None

    for key, value in pairs.items():
        if max_pair is None or value > pairs[max_pair]:
            max_pair = key

    return max_pair


In [6]:
def apply_merge(word_freq, max_pair):
    new_freq = dict()
    for word in word_freq:
        new_word = []
        i = 0
        while i < (len(word)):
            if i < len(word)-1 and (word[i],word[i+1]) == max_pair:
                new_word.append(word[i] + word[i+1])
                i+=1
            else:
                new_word.append(word[i])
            i += 1
        new_freq[tuple(new_word)] = word_freq[word]
    return new_freq

In [13]:
corpus = [
    "low lower lowest",
    "low lower",
    "new newer newest",
    "new news",
    "wide wider widest",
    "wide widely",
    "small smaller smallest",
    "small smart",
    "smart smarter smartest",
    "fast faster fastest",
]

def bytePairEncoding(corpus, MAX_MERGES):
    merge_rules = []

    corpus = normalize(corpus)
    word_freq = corpus_to_word_frequency(corpus)

    vocab = generate_initial_vocab(word_freq)

    while len(merge_rules) < MAX_MERGES:
        pairs = count_pairs(word_freq)
        if len(pairs) == 0: 
            break
        max_pair = get_frequent_pair(pairs)
        merge_rules.append(max_pair)
        if (max_pair[0] + max_pair[1]) not in vocab:
            vocab.append(max_pair[0] + max_pair[1])    
        word_freq = apply_merge(word_freq, max_pair)
    return word_freq, merge_rules, vocab

corpus, merge_rules, vocab = bytePairEncoding(corpus, 15)

print(merge_rules)
print(vocab)

[('s', 't'), ('s', 'm'), ('sm', 'a'), ('e', 'r'), ('er', '</w>'), ('st', '</w>'), ('e', 'st</w>'), ('l', 'o'), ('lo', 'w'), ('n', 'e'), ('ne', 'w'), ('w', 'i'), ('wi', 'd'), ('sma', 'l'), ('smal', 'l')]
['l', 'o', 'w', '</w>', 'e', 'r', 's', 't', 'n', 'i', 'd', 'y', 'm', 'a', 'f', 'st', 'sm', 'sma', 'er', 'er</w>', 'st</w>', 'est</w>', 'lo', 'low', 'ne', 'new', 'wi', 'wid', 'smal', 'small']


In [14]:
new_sentence = ["The cat walks quickly through the low enclosure. It feels smart as it sees new, small rabbits eat wide carrots"]

def tokenize(sentence, merge_rules):
    sentence = normalize(sentence)
    for merge in merge_rules:
        sentence = apply_merge(sentence, merge)

    return sentence


In [15]:
import pandas as pd

df = pd.read_csv("spa-eng/spa.txt", sep='\t')
df = df.drop(df.columns[2], axis = 1)
corpus = df.values.ravel().tolist()

In [16]:
corpus, merge_rules, vocab = bytePairEncoding(corpus, MAX_MERGES=500)